# Example 12.6.3 / 2: Explicit nonanticipativity

Companion notebook for **12.6.3 Two-Stage Recourse and Nonanticipativity** in the AMPL lab Documentation, section "Explicit nonanticipativity". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Explicit nonanticipativity

Scenario-indexed copies of early decisions are sometimes useful in multistage or decomposition formulations. Copies belonging to scenarios with the same history must be linked by equality constraints. For a true scenario tree, link only nodes that share the same observed history at that stage.

The order model with one copy per scenario shows what `SameFirstStage` enforces. The run solves
with the links, then removes them with the amplpy `drop()` method (`drop SameFirstStage;` in AMPL):

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set SCENARIOS ordered;               # demand outcomes; the first anchors the copies
param probability {SCENARIOS} >= 0;  # scenario weight
param demand {SCENARIOS} >= 0;       # units
param unit_cost >= 0;                # dollars per unit ordered
param price >= 0;                    # dollars per unit sold

# One copy of the order per scenario.
var OrderCopy {SCENARIOS} >= 0;
var Sell {SCENARIOS} >= 0;           # second stage: sales per scenario

maximize ExpectedProfit:             # expected profit over the scenario copies
    sum {s in SCENARIOS} probability[s]
        * (price * Sell[s] - unit_cost * OrderCopy[s]);

subject to SellFromOrder {s in SCENARIOS}: Sell[s] <= OrderCopy[s];  # sales from this scenario copy
# sales capped by the revealed demand
subject to SellToDemand {s in SCENARIOS}: Sell[s] <= demand[s];

# Nonanticipativity: every copy equals the first scenario's copy.
subject to SameFirstStage {s in SCENARIOS diff {first(SCENARIOS)}}:
    OrderCopy[s] = OrderCopy[first(SCENARIOS)];

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["SCENARIOS"] = ["low", "mid", "high"]
ampl.param["probability"] = {"low": 0.3, "mid": 0.4, "high": 0.3}
ampl.param["demand"] = {"low": 40, "mid": 100, "high": 130}
ampl.param["unit_cost"] = 4
ampl.param["price"] = 10

In [5]:
# run.py
ampl.solve(solver="highs", verbose=False)
print("linked:", ampl.var["OrderCopy"].to_dict(), ampl.obj["ExpectedProfit"].value())

ampl.con["SameFirstStage"].drop()
ampl.solve(solver="highs", verbose=False)
print("unlinked:", ampl.var["OrderCopy"].to_dict(), ampl.obj["ExpectedProfit"].value())

linked: {'low': 100, 'mid': 100, 'high': 100} 420.0
unlinked: {'low': 40, 'mid': 100, 'high': 130} 546.0


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set SCENARIOS := low mid high;
param probability := low 0.3  mid 0.4  high 0.3;
param demand := low 40  mid 100  high 130;
param unit_cost := 4;
param price := 10;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display OrderCopy, ExpectedProfit;
drop SameFirstStage;
solve;
display OrderCopy, ExpectedProfit;

HiGHS 1.14.0: optimal solution; objective 420
1 simplex iterations
0 barrier iterations
OrderCopy [*] :=
 low  100
 mid  100
high  100
;

ExpectedProfit = 420

HiGHS 1.14.0: optimal solution; objective 546
2 simplex iterations
0 barrier iterations
OrderCopy [*] :=
 low   40
 mid  100
high  130
;

ExpectedProfit = 546

